In [1]:
import os
import pandas as pd
from datetime import datetime
from working_data import clean_five_minute_data, add_timing, normalize_by_window, clean_hour_data, split_multiresolution_chunks, regression_label_df
from constants.global_constants import *

In [2]:
starting_dir = "data/finer_data"
working_path = "data/experimenting"
instrument = "GBPUSD#"


In [3]:
df = pd.read_csv(f"{starting_dir}/{instrument}/five_minutes.csv")
print(f"Processing {instrument}...")
print(f"  5 minute Original data: {len(df)} rows")
print(f"  5 minute Time range: {datetime.fromtimestamp(df['time'].min())} to {datetime.fromtimestamp(df['time'].max())}")
df = clean_five_minute_data(df)
print(f"  Cleaned data: {len(df)} rows")
df = add_timing(df)
df = normalize_by_window(
    df, 
    window_size=NORMALIZING_WINDOW_SIZE, 
    low_col='low',
    high_col='high',
    normalizing_cols=[
        'open',
        'high',
        'low',
        'close'
    ],
    label_cols=['open', 'close'])

hour_df = pd.read_csv(f"{starting_dir}/{instrument}/hours.csv")
print(f"  Hour Original data: {len(hour_df)} rows")
print(f"  Hour Time range: {datetime.fromtimestamp(hour_df['time'].min())} to {datetime.fromtimestamp(hour_df['time'].max())}")
hour_df = clean_hour_data(hour_df)
print(f"  Cleaned data: {len(hour_df)} rows")
hour_df = normalize_by_window(
    hour_df, 
    window_size=NORMALIZING_WINDOW_SIZE, 
    low_col='low',
    high_col='high',
    normalizing_cols=[
        'open',
        'high',
        'low',
        'close'
    ],
    label_cols=['open', 'close'])

print(f"Labeling...\n\n\n")
df = regression_label_df(df, window_size=REGRESSION_LABELING_WINDOW_SIZE, 
                positive_slope=POSITIVE_SLOPE, 
                negative_slope=NEGATIVE_SLOPE,
                starting_hour=STARTING_HOUR,
                ending_hour=ENDING_HOUR,
                lookback_window=LABEL_LOOKBACK)


os.makedirs(f"{working_path}/{instrument}", exist_ok=True)

hour_df.to_csv(f"{working_path}/{instrument}/hour.csv", index=False)
split_multiresolution_chunks(df_5min=df,
                            df_hour=hour_df,
                            dump_path=f"{working_path}/{instrument}",
                            chunk_size=20000,
                            hour_lookback=OTHER_TOKENS,
                            lookback=NUM_TOKENS,
                            cols=[
                                'time',
                                'open_normalized',
                                'high_normalized',
                                'low_normalized',
                                'close_normalized',
                                'include',
                                'target_high',
                                'target_low'
                            ])

Processing GBPUSD#...
  5 minute Original data: 78298 rows
  5 minute Time range: 2024-06-01 00:00:00 to 2025-06-20 21:15:00
  Cleaned data: 78236 rows
  Hour Original data: 6528 rows
  Hour Time range: 2024-06-01 00:00:00 to 2025-06-20 21:00:00
  Cleaned data: 6469 rows
Labeling...





{'total_chunks': 4,
 'total_original_rows': 78086,
 'chunks_info': [{'chunk_num': 0,
   'start_idx': 0,
   'end_idx': 20000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 1717409400,
   'end_time': 1725659400,
   'hour_start_pos': -1},
  {'chunk_num': 1,
   'start_idx': 20000,
   'end_idx': 40000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 1725659700,
   'end_time': 1734082800,
   'hour_start_pos': 1519},
  {'chunk_num': 2,
   'start_idx': 40000,
   'end_idx': 60000,
   'total_rows': 20000,
   'train_rows': 13999,
   'val_rows': 2936,
   'test_rows': 2937,
   'start_time': 1734083100,
   'end_time': 1742935800,
   'hour_start_pos': 3173},
  {'chunk_num': 3,
   'start_idx': 60000,
   'end_idx': 78086,
   'total_rows': 18086,
   'train_rows': 12660,
   'val_rows': 2649,
   'test_rows': 2649,
   'start_time': 1742936100,
   'end_time': 1750445100,
   'hour_start_p